# Pré-processamento do Dataset de Elementos Transponíveis

In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

## 1. Caminhos de entrada e saída

In [2]:
input_file = "../data/TE_dataset_final_clean.csv"
output_file = "../data/TE_dataset_final_pre_processed.csv"
output_dir = os.path.dirname(output_file)

print("🔹 Iniciando o pré-processamento...")
print(f"Carregando dataset: {input_file}")

🔹 Iniciando o pré-processamento...
Carregando dataset: ../data/TE_dataset_final_clean.csv


## 2. Carregar o dataset bruto

In [3]:
df = pd.read_csv(input_file)
tamanho_original = len(df)
print(f"Tamanho original: {tamanho_original:,} sequências")

Tamanho original: 337,672 sequências


## 3. Limpeza de dados

In [4]:
print("\n Iniciando pipeline de limpeza...")

# Garantir que exista coluna de comprimento
if "Comprimento" not in df.columns:
    df["Comprimento"] = df["Sequência de TE"].str.len()

# Máscaras de filtragem
seq_vazias_ou_nan = df["Sequência de TE"].isna() | (df["Sequência de TE"] == "")
seq_com_n = df["Sequência de TE"].str.contains("N", na=False)
seq_muito_curtas = df["Comprimento"] < 50

print(f"  • Removendo {seq_vazias_ou_nan.sum()} sequências NaN/vazias")
print(f"  • Removendo {seq_com_n.sum()} sequências com 'N'")
print(f"  • Removendo {seq_muito_curtas.sum()} sequências < 50bp")

# Aplicar máscara
mascara_valida = ~seq_vazias_ou_nan & ~seq_com_n & ~seq_muito_curtas
df_clean = df[mascara_valida].copy()

# Estatísticas de limpeza
tamanho_final = len(df_clean)
removidas = tamanho_original - tamanho_final
print(f"\n✅ Limpeza concluída: {removidas:,} removidas | {tamanho_final:,} restantes")


 Iniciando pipeline de limpeza...
  • Removendo 0 sequências NaN/vazias
  • Removendo 0 sequências com 'N'
  • Removendo 5236 sequências < 50bp

✅ Limpeza concluída: 5,236 removidas | 332,436 restantes


## 4. Engenharia de Features

In [5]:
print("\nCriando feature 'Comprimento_Log'...")

df_clean["Comprimento_Log"] = np.log1p(df_clean["Comprimento"])
df_clean.drop(columns=["Comprimento"], inplace=True)

print("  • 'Comprimento_Log' criada com sucesso.")
print("  • 'Comprimento' original removida.")


Criando feature 'Comprimento_Log'...
  • 'Comprimento_Log' criada com sucesso.
  • 'Comprimento' original removida.


# 5. Salvar dataset final

In [6]:
os.makedirs(output_dir, exist_ok=True)
df_clean.to_csv(output_file, index=False)

print(f"\n Dataset salvo em: {output_file}")
print("===================================================")
print("Pré-processamento concluído com sucesso!")
print("===================================================")

# (opcional) Mostrar primeiras linhas e estatísticas
print("\nPrévia do dataset limpo:")
print(df_clean.head())

print("\nEstatísticas de 'Comprimento_Log':")
print(df_clean["Comprimento_Log"].describe())


 Dataset salvo em: ../data/TE_dataset_final_pre_processed.csv
Pré-processamento concluído com sucesso!

Prévia do dataset limpo:
  Cromossomo                                    Sequência de TE    Classe  \
0          7  GAGCTTCGTCACCAGCTTTGCTCCGACCACCCTTTGTCCATACTAA...  Helitron   
1          1  GTCAGGGTTGCTTCTTGGCGAAGACAGGGCCTCGGGCGAGCCAGAA...  Helitron   
2          8  ACGCCCAAGCAGACGGTCACCATCAGCGAAGACCTCACTTCGCATG...  Helitron   
3          5  TATGCCAAGTCGTGTCAAACGACTTAGGGTAGGGGTCAACTTTCTC...  Helitron   
4          9  GTTAGGTTATTTATATACTAGTTTATGTTGATGATATAATCATCAC...  Helitron   

   Comprimento_Log  
0         7.288928  
1         5.899897  
2         5.662960  
3         5.327876  
4         8.410053  

Estatísticas de 'Comprimento_Log':
count    332436.000000
mean          6.788389
std           1.795069
min           3.931826
25%           5.236442
50%           6.369901
75%           8.409385
max          11.572250
Name: Comprimento_Log, dtype: float64
